# 🛰️ Copernicus Climate Data Store (CDS) - Exemplos

Este notebook demonstra como usar o accessor do Copernicus Climate Data Store (CDS) para obter dados climáticos para modelagem epidemiológica.

## 📋 Conteúdo

1. [Setup e Configuração](#setup)
2. [Listar Variáveis e Áreas Disponíveis](#listar)
3. [Download de Dados Climáticos](#download)
4. [Análise de Temperatura para o Brasil](#temperatura)
5. [Análise de Precipitação](#precipitacao)
6. [Agregação Temporal](#agregacao)
7. [Correlação com Dados Epidemiológicos](#correlacao)
8. [Dados em Escala Municipal](#cidade)

## 📖 Documentação

- **CDS Website**: https://cds.climate.copernicus.eu/
- **CDS API Docs**: https://cds.climate.copernicus.eu/how-to-api
- **ERA5 Dataset**: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels

<a id='setup'></a>
## 1. Setup e Configuração

### Pré-requisitos

1. **Instalar dependências**:
   ```bash
   pip install cdsapi xarray netCDF4 pandas matplotlib
   ```

2. **Criar conta no CDS**:
   - Acesse https://cds.climate.copernicus.eu/
   - Registre-se gratuitamente

3. **Configurar credenciais**:
   - Crie o arquivo `~/.cdsapirc` com:
   ```
   url: https://cds.climate.copernicus.eu/api
   key: SUA_API_KEY_AQUI
   ```
   - Ou use variáveis de ambiente: `CDSAPI_URL` e `CDSAPI_KEY`

4. **Aceitar os Termos de Uso**:
   - Acesse a página do dataset ERA5
   - Aceite os termos de uso antes de fazer download

In [3]:
# Adicionar o path do projeto

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Verificar se as dependências estão instaladas
try:
    import cdsapi
    print("✅ cdsapi instalado")
except ImportError:
    print("❌ Instale cdsapi: pip install cdsapi")

try:
    import xarray as xr
    print("✅ xarray instalado")
except ImportError:
    print("❌ Instale xarray: pip install xarray netCDF4")

✅ cdsapi instalado
✅ xarray instalado


In [4]:
# Importar o accessor
from epidatasets.sources.copernicus_cds import CopernicusCDSAccessor

# Inicializar
try:
    cds = CopernicusCDSAccessor()
    print("✅ CDS accessor inicializado com sucesso!")
except Exception as e:
    print(f"❌ Erro na inicialização: {e}")

❌ Erro na inicialização: CDS API credentials not found.
Please configure in one of:
  - Environment variables: CDSAPI_URL, CDSAPI_KEY
  - Config file: ~/.cdsapirc
  - Config file: ~/.nanobot/config/cdsapi.env

To get credentials:
  1. Register at https://cds.climate.copernicus.eu/
  2. Go to your profile page
  3. Copy the API key
  4. Create ~/.cdsapirc with:
     url: https://cds.climate.copernicus.eu/api
     key: YOUR_API_KEY


<a id='listar'></a>
## 2. Listar Variáveis e Áreas Disponíveis

In [ ]:
# Listar variáveis disponíveis
variables = cds.list_variables()
print("📊 Variáveis disponíveis para modelagem epidemiológica:")
variables.head(15)

In [ ]:
# Listar áreas pré-definidas
areas = cds.list_areas()
print("🌍 Áreas pré-definidas:")
areas

In [ ]:
# Listar datasets disponíveis
datasets = cds.list_datasets()
print("📁 Datasets disponíveis:")
datasets

<a id='download'></a>
## 3. Download de Dados Climáticos

**⚠️ Aviso**: Downloads do CDS podem demorar vários minutos dependendo do volume de dados solicitado. Os dados são armazenados em cache para reutilização.

In [ ]:
# Exemplo: Download de temperatura para o Brasil (período curto para demonstração)
# Nota: Use períodos maiores para análises reais

try:
    print("🌡️ Baixando dados de temperatura para o Brasil...")
    print("⏳ Isso pode levar alguns minutos...")
    
    # Usando período curto para exemplo (3 dias)
    temp_data = cds.get_temperature(
        start_date='2024-01-01',
        end_date='2024-01-03',
        area='brazil',
        use_cache=True
    )
    
    print("✅ Download concluído!")
    print(f"\n📊 Dimensões do dataset:")
    print(temp_data)
    
except Exception as e:
    print(f"❌ Erro no download: {e}")
    print("\n💡 Dica: Verifique se aceitou os Termos de Uso do dataset ERA5")

<a id='temperatura'></a>
## 4. Análise de Temperatura para o Brasil

In [ ]:
# Visualizar estrutura dos dados
if 'temp_data' in locals():
    print("📊 Estrutura do dataset:")
    print(temp_data.info())

In [ ]:
# Converter para DataFrame (média espacial)
if 'temp_data' in locals():
    # Calcular média espacial
    temp_mean = temp_data.mean(dim=['latitude', 'longitude'])
    temp_df = temp_mean.to_dataframe().reset_index()
    
    print("🌡️ Temperatura média do Brasil (série temporal):")
    print(temp_df.head(10))

In [ ]:
# Plotar série temporal de temperatura
if 'temp_df' in locals() and not temp_df.empty:
    plt.figure(figsize=(12, 5))
    
    # Converter de Kelvin para Celsius
    temp_celsius = temp_df['t2m'] - 273.15
    
    plt.plot(temp_df['time'], temp_celsius, linewidth=1)
    plt.title('Temperatura Média do Brasil (2m acima do solo)', fontsize=14)
    plt.xlabel('Data/Hora')
    plt.ylabel('Temperatura (°C)')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Mapa de temperatura média (média temporal)
if 'temp_data' in locals():
    temp_time_mean = temp_data.mean(dim='time')
    
    plt.figure(figsize=(10, 8))
    temp_time_mean['t2m'].plot(
        cmap='RdYlBu_r',
        vmin=temp_time_mean['t2m'].min(),
        vmax=temp_time_mean['t2m'].max(),
        cbar_kwargs={'label': 'Temperatura (K)'}
    )
    plt.title('Temperatura Média - Brasil', fontsize=14)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.show()

<a id='precipitacao'></a>
## 5. Análise de Precipitação

In [ ]:
# Download de precipitação
try:
    print("🌧️ Baixando dados de precipitação para o Brasil...")
    
    precip_data = cds.get_precipitation(
        start_date='2024-01-01',
        end_date='2024-01-03',
        area='brazil',
        use_cache=True
    )
    
    print("✅ Download concluído!")
    
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Agregar precipitação diária
if 'precip_data' in locals():
    precip_daily = cds.aggregate_to_daily(precip_data, method='sum')
    precip_mean = precip_daily.mean(dim=['latitude', 'longitude'])
    precip_df = precip_mean.to_dataframe().reset_index()
    
    print("🌧️ Precipitação diária acumulada (média Brasil):")
    print(precip_df)

In [ ]:
# Plotar precipitação diária
if 'precip_df' in locals() and not precip_df.empty:
    plt.figure(figsize=(10, 5))
    
    # Converter de m para mm
    precip_mm = precip_df['tp'] * 1000
    
    plt.bar(precip_df['time'].dt.strftime('%Y-%m-%d'), precip_mm, color='steelblue')
    plt.title('Precipitação Diária - Brasil', fontsize=14)
    plt.xlabel('Data')
    plt.ylabel('Precipitação (mm)')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

<a id='agregacao'></a>
## 6. Agregação Temporal

In [ ]:
# Demonstração de diferentes agregações temporais
if 'temp_data' in locals():
    # Agregar para valores diários
    temp_daily_max = cds.aggregate_to_daily(temp_data, method='max')
    temp_daily_min = cds.aggregate_to_daily(temp_data, method='min')
    temp_daily_mean = cds.aggregate_to_daily(temp_data, method='mean')
    
    # Média espacial
    max_vals = temp_daily_max.mean(dim=['latitude', 'longitude']).to_dataframe()
    min_vals = temp_daily_min.mean(dim=['latitude', 'longitude']).to_dataframe()
    mean_vals = temp_daily_mean.mean(dim=['latitude', 'longitude']).to_dataframe()
    
    # Plotar
    plt.figure(figsize=(12, 5))
    
    plt.plot(max_vals.index, max_vals['t2m'] - 273.15, 'r-', label='Máxima', linewidth=2)
    plt.plot(mean_vals.index, mean_vals['t2m'] - 273.15, 'g-', label='Média', linewidth=2)
    plt.plot(min_vals.index, min_vals['t2m'] - 273.15, 'b-', label='Mínima', linewidth=2)
    
    plt.fill_between(max_vals.index, max_vals['t2m'] - 273.15, min_vals['t2m'] - 273.15, 
                     alpha=0.2, color='gray', label='Amplitude')
    
    plt.title('Temperatura Diária - Brasil (Máx, Méd, Mín)', fontsize=14)
    plt.xlabel('Data')
    plt.ylabel('Temperatura (°C)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

<a id='correlacao'></a>
## 7. Correlação com Dados Epidemiológicos

Exemplo de como integrar dados climáticos com dados de dengue.

In [ ]:
# Simular dados de dengue (em uma aplicação real, viriam de outro accessor)
if 'temp_df' in locals():
    # Criar datas diárias
    dates = pd.date_range('2024-01-01', '2024-01-03', freq='D')
    
    # Simular casos de dengue com correlação com temperatura
    np.random.seed(42)
    base_cases = 100
    temp_factor = np.array([1.2, 1.5, 1.3])  # Mais casos em dias quentes
    dengue_cases = (base_cases * temp_factor + np.random.normal(0, 10, 3)).astype(int)
    
    dengue_df = pd.DataFrame({
        'date': dates,
        'cases': dengue_cases
    })
    
    print("🦟 Dados simulados de dengue:")
    print(dengue_df)

In [ ]:
# Combinar dados climáticos e epidemiológicos
if 'temp_df' in locals() and 'dengue_df' in locals():
    # Agregar temperatura para diário
    temp_daily = cds.aggregate_to_daily(temp_data, method='mean')
    temp_daily_mean = temp_daily.mean(dim=['latitude', 'longitude'])
    temp_daily_df = temp_daily_mean.to_dataframe().reset_index()
    temp_daily_df['date'] = temp_daily_df['time'].dt.date
    
    # Converter temperatura para Celsius
    temp_daily_df['temp_celsius'] = temp_daily_df['t2m'] - 273.15
    
    # Juntar dados
    dengue_df['date'] = pd.to_datetime(dengue_df['date']).dt.date
    combined = pd.merge(dengue_df, temp_daily_df[['date', 'temp_celsius']], on='date')
    
    print("📊 Dados combinados (Clima + Dengue):")
    print(combined)

In [ ]:
# Visualizar correlação
if 'combined' in locals() and not combined.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    
    # Plot 1: Temperatura
    ax1.plot(combined['date'], combined['temp_celsius'], 'r-o', linewidth=2, markersize=8)
    ax1.set_ylabel('Temperatura (°C)', color='red')
    ax1.tick_params(axis='y', labelcolor='red')
    ax1.grid(True, alpha=0.3)
    ax1.set_title('Correlação Temperatura vs Casos de Dengue', fontsize=14)
    
    # Plot 2: Casos de dengue
    ax2.bar(combined['date'].astype(str), combined['cases'], color='steelblue', alpha=0.7)
    ax2.set_ylabel('Casos de Dengue', color='steelblue')
    ax2.tick_params(axis='y', labelcolor='steelblue')
    ax2.set_xlabel('Data')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Calcular correlação
    corr = combined['temp_celsius'].corr(combined['cases'])
    print(f"\n📈 Correlação temperatura-casos: {corr:.3f}")

## 📚 Resumo

Este notebook demonstrou:

1. ✅ **Configuração** do accessor Copernicus CDS
2. ✅ **Listagem** de variáveis e áreas disponíveis
3. ✅ **Download** de dados climáticos (temperatura, precipitação)
4. ✅ **Visualização** espacial e temporal
5. ✅ **Agregação** de dados horários para diário/semanal
6. ✅ **Integração** com dados epidemiológicos

## 💡 Dicas para Análises Reais

- Use períodos mais longos (mínimo 1 ano) para análises sazonais
- Considere lag temporal (ex: temperatura afeta dengue 2-3 semanas depois)
- Combine múltiplas variáveis (temperatura + precipitação + umidade)
- Use dados municipais quando disponíveis para análises locais

## 🔗 Recursos Adicionais

- [Documentação CDS](https://cds.climate.copernicus.eu/)
- [ERA5 Data Documentation](https://confluence.ecmwf.int/display/CKB/ERA5)
- [cdsapi GitHub](https://github.com/ecmwf/cdsapi)

<a id='cidade'></a>
## 8. Dados em Escala Municipal

O accessor agora suporta busca de dados climáticos para cidades e municípios brasileiros. Isso é útil para:
- Análise de correlação clima-dengue em escala local
- Modelos preditivos municipais
- Estudos epidemiológicos regionais

In [ ]:
# Listar cidades disponíveis
cities = cds.list_cities()
print(f"📍 Total de cidades disponíveis: {len(cities)}")
print("\nPrimeiras 15 cidades:")
cities.head(15)

In [ ]:
# Listar cidades por estado (exemplo: São Paulo)
cities_sp = cds.list_cities(state='SP')
print(f"📍 Cidades de São Paulo disponíveis: {len(cities_sp)}")
cities_sp.head(10)

In [ ]:
# Obter bounding box para uma cidade
bbox_sp = cds.get_city_bounding_box('sao_paulo', buffer_km=30)
print(f"🗺️ Bounding box para São Paulo (30km buffer):")
print(f"   Norte: {bbox_sp[0]:.4f}")
print(f"   Oeste: {bbox_sp[1]:.4f}")
print(f"   Sul:   {bbox_sp[2]:.4f}")
print(f"   Leste: {bbox_sp[3]:.4f}")

# Também pode usar código IBGE
bbox_rj = cds.get_city_bounding_box('3304557', buffer_km=25)  # Rio de Janeiro
print(f"\n🗺️ Bounding box para Rio de Janeiro (25km buffer):")
print(f"   Norte: {bbox_rj[0]:.4f}")
print(f"   Oeste: {bbox_rj[1]:.4f}")
print(f"   Sul:   {bbox_rj[2]:.4f}")
print(f"   Leste: {bbox_rj[3]:.4f}")

In [ ]:
# Download de dados para uma cidade específica
# Nota: Usando período curto para demonstração

try:
    print("🌡️ Baixando dados de temperatura para São Paulo...")
    
    temp_sp = cds.get_city_temperature(
        city='sao_paulo',
        start_date='2024-01-01',
        end_date='2024-01-03',
        buffer_km=30,  # 30km ao redor do centro
        use_cache=True
    )
    
    print("✅ Download concluído!")
    print(f"\n📊 Dimensões: {temp_sp.dims}")
    
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Obter série temporal limpa para uma cidade
# Este método já faz a agregação espacial e temporal automaticamente

try:
    print("📈 Obtendo série temporal de temperatura para São Paulo...")
    
    # Dados diários agregados
    df_sp = cds.get_city_timeseries(
        city='sao_paulo',
        variable='2m_temperature',
        start_date='2024-01-01',
        end_date='2024-01-03',
        buffer_km=30,
        aggregation='daily',
        spatial_agg='mean',
    )
    
    # Converter Kelvin para Celsius
    df_sp['temp_celsius'] = df_sp['value'] - 273.15
    
    print("✅ Série temporal obtida!")
    print(df_sp)
    
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Comparar temperatura entre múltiplas cidades

try:
    cities_to_compare = ['sao_paulo', 'rio_de_janeiro', 'salvador']
    city_data = {}
    
    for city in cities_to_compare:
        print(f"📊 Obtendo dados para {city}...")
        df = cds.get_city_timeseries(
            city=city,
            variable='2m_temperature',
            start_date='2024-01-01',
            end_date='2024-01-03',
            buffer_km=30,
            aggregation='daily',
        )
        df['temp_celsius'] = df['value'] - 273.15
        city_data[city.replace('_', ' ').title()] = df
    
    # Plotar comparação
    plt.figure(figsize=(12, 6))
    
    for city_name, df in city_data.items():
        plt.plot(df['date'], df['temp_celsius'], marker='o', label=city_name, linewidth=2)
    
    plt.title('Comparação de Temperatura entre Cidades', fontsize=14)
    plt.xlabel('Data')
    plt.ylabel('Temperatura (°C)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"❌ Erro: {e}")

### 💡 Recursos de Escala Municipal

O accessor inclui **70+ cidades brasileiras** pré-configuradas:

- **Todas as capitais estaduais** (27 cidades)
- **Principais municípios** com alta incidência de dengue
- **Suporte a código IBGE** para integração com outros datasets

**Métodos disponíveis:**
- `list_cities()` - Lista cidades disponíveis
- `get_city_bounding_box()` - Obtém bounding box para uma cidade
- `get_city_data()` - Busca dados climáticos para uma cidade
- `get_city_temperature()` - Temperatura específica para cidade
- `get_city_precipitation()` - Precipitação específica para cidade
- `get_city_timeseries()` - Série temporal limpa (pronta para análise)

**Parâmetro `buffer_km`:**
- Define o raio ao redor do centro da cidade
- Padrão: 50km (adequado para municípios grandes)
- Use valores menores (20-30km) para cidades compactas
- Use valores maiores (100km+) para regiões metropolitanas

In [ ]:
## 📚 Resumo

Este notebook demonstrou:

1. ✅ **Configuração** do accessor Copernicus CDS
2. ✅ **Listagem** de variáveis e áreas disponíveis
3. ✅ **Download** de dados climáticos (temperatura, precipitação)
4. ✅ **Visualização** espacial e temporal
5. ✅ **Agregação** de dados horários para diário/semanal
6. ✅ **Integração** com dados epidemiológicos
7. ✅ **Dados em escala municipal** - 70+ cidades brasileiras

## 💡 Dicas para Análises Reais

- Use períodos mais longos (mínimo 1 ano) para análises sazonais
- Considere lag temporal (ex: temperatura afeta dengue 2-3 semanas depois)
- Combine múltiplas variáveis (temperatura + precipitação + umidade)
- Use `buffer_km` adequado para o tamanho do município
- Integre com códigos IBGE para análises padronizadas

## 🔗 Recursos Adicionais

- [Documentação CDS](https://cds.climate.copernicus.eu/)
- [ERA5 Data Documentation](https://confluence.ecmwf.int/display/CKB/ERA5)
- [cdsapi GitHub](https://github.com/ecmwf/cdsapi)